This file shows how to fit a simple sine function within $[-2\pi, 2\pi]$ using neural networks.

### Step 0: Setups
---

In [ ]:
# To allow automatically uses the newest version of your code.  
%load_ext autoreload
%autoreload 2

In [ ]:
# Add necessary paths to the system path
from utils import add_necessary_paths
add_necessary_paths()

# Import libraries
from utils import check_torch

import utils.u01 as u
import talos as ta
import torch
import numpy as np

# Do sanity checks before running the code
check_torch()

# Fix random seed for reproducibility
# NOTE: Run cells IN ORDER to ensure deterministic behavior.
ta.set_seed()

### Step 1: Prepare data
---

In [ ]:
# configure data set  
t_min, t_max = -2 * np.pi, 2 * np.pi
n_points = 200
shuffle = 1
train_test_ratio = (1, 1)

# Generate data
X = np.linspace(t_min, t_max, n_points).reshape(-1, 1)  # Reshape to a column vector
Y = np.sin(X).reshape(-1, 1)  # Reshape to a column vector
dataset = ta.Dataset(X, Y, name='sin(x)')
dataset.report()

# Split data
train_set, test_set = dataset.split(*train_test_ratio, shuffle=shuffle)
train_set.report()
test_set.report()

# Visualize data
ax = u.get_axis((6, 3))
u.plot_points(ax, train_set.X, train_set.Y, 'blue', 'Train Set')
u.plot_points(ax, test_set.X, test_set.Y, 'green', 'Test Set')
u.finalize_plot(ax, title=f'Dataset Visualization (shuffle = {shuffle})')

### Step 2: Build model
---

In [ ]:
# Model configuration  
hidden_features = [50, 50, 20, 50, 50]

# Instantiate a model
model = ta.model.torch_zoo.MLP(1, hidden_features, 1, activation='tanh')
print(model)

### Step 3: Train and evaluate model
---

In [ ]:
# Configure training parameters  
trainer = ta.TorchTrainer(model, loss_fn='mse')
trainer.config.early_stop = True
trainer.config.patience = 2
trainer.config.val_ratio = 0.1
trainer.config.validate_every = 10000
trainer.config.print_every = 5000

# Train the model  
trainer.train(train_set, 50000)

In [ ]:
# Evaluate the model on the training set
Y_pred = model.predict(test_set)

# Visualize the model prediction against the ground truth
ax = u.get_axis((6, 3))
u.plot_points(ax, test_set.X, test_set.Y, 'green', 'Ground Truth')
u.plot_points(ax, test_set.X, Y_pred, 'red', 'Prediction')
u.finalize_plot(ax, title='Model Prediction vs Ground Truth (Test Set)')

## What happens if we let shuffle=0 when splitting the dataset?

In [ ]:
# Re-split dataset without shuffling and visualize
shuffle = 0
train_set_ns, test_set_ns = dataset.split(*train_test_ratio, shuffle=shuffle)
train_set_ns.report()
test_set_ns.report()

ax = u.get_axis((6, 3))
u.plot_points(ax, train_set_ns.X, train_set_ns.Y, 'blue', 'Train Set (no shuffle)')
u.plot_points(ax, test_set_ns.X, test_set_ns.Y, 'green', 'Test Set (no shuffle)')
u.finalize_plot(ax, title=f'Dataset Visualization (shuffle = {shuffle})')

In [ ]:
# Initialize another model and train it
model_alt = ta.model.torch_zoo.MLP(1, hidden_features, 1, activation='tanh')

trainer_alt = ta.TorchTrainer(model_alt, loss_fn='mse')
trainer_alt.config.early_stop = True
trainer_alt.config.patience = 2
trainer_alt.config.val_ratio = 0.1
trainer_alt.config.validate_every = 10000
trainer_alt.config.print_every = 5000

trainer_alt.train(train_set_ns, 50000)

In [ ]:
# Evaluate and visualize
Y_pred_alt = model_alt.predict(test_set_ns)
ax = u.get_axis((6, 3))
u.plot_points(ax, test_set_ns.X, test_set_ns.Y, 'green', 'Ground Truth')
u.plot_points(ax, test_set_ns.X, Y_pred_alt, 'orange', 'Prediction (alt model)')
u.finalize_plot(ax, title='Alt Model Prediction vs Ground Truth (Test Set (ns))')